# Scenario 1: <sup>1</sup>H-NMR kinetic analysis of Pd(0)-catalyzed cycloisomerization

## Data Modelling Notebook

---

### Notebook Setup

The essential packages to work with the modelling Notebook are:

- `catalax`
- `pyenzyme`

The following packages in this Notebook are used for convenience:

- `math`
- `pathlib`
- `rich`

Depending on the use case, some additional packages maybe useful:

- `numpy`
- `numpyro`
- `jax.numpy`
- `...`

In [ ]:
import numpyro
numpyro.set_host_device_count(8)

In [ ]:
from math import e
from pathlib import Path
from rich import print

import catalax as ctx
import catalax.mcmc as mcmc
import jax.numpy as jnp
import pyenzyme as pe

---

### Setting up the modelling environment

Besides loading the EnzymeML document from the previous Notebook, we also need to set up the modelling environment. This includes defining the model, the parameters to be estimated, and the data to be used for fitting.

#### EnzymeML document

In [ ]:
path_to_enzymeml = Path.cwd() / "data/output/cycloisomerization.json"

In [ ]:
doc = pe.read_enzymeml(path_to_enzymeml)

The species not relevant to the modelling can be deleted so they do not interfere with the fitting process and do not show up in the plots.

In [ ]:
id_to_delete = [1, 4, 5]

for i in sorted(id_to_delete, reverse=True):
    del doc.small_molecules[i]
    
for m in doc.measurements:
    for i in sorted(id_to_delete, reverse=True):
        del m.species_data[i]

In [ ]:
pe.summary(doc)

#### Catalax dataset

Due to the way Catalax works, if species are relevant to another species' ODE, but do not receive an ODE themselves, they have to be removed from the dataset.

Here, we pop both Pd(0) (s3) and BINOL (s4) from the dataset's states and from each measurement's states and data.

In [ ]:
dataset = ctx.Dataset.from_enzymeml(doc)
dataset = dataset.pad()

In [ ]:
dataset.states.pop()
dataset.states.pop()
for m in dataset.measurements:
    m.data.popitem()
    m.data.popitem()
    m.state.pop()
    m.state.pop()

In [ ]:
dataset.plot(show=True)

#### Catalax model

Here, the model is defined in form an ODE. Additionally, the equilibrium constant at 80°C is calculated from the one at 20°C and the thermodynamic parameters. Finally, uniform priors are set for the parameters to be estimated.

In [ ]:
model = ctx.Model(name="Pseudo 1st order irreversible with background")
model.add_state("s1")
model.add_constants(s3="s3", s4="s4")
model

KEQ_20 = 8.8
R = 1.987    # cal/mol*K
dH = -10200  # cal/mol
dS = -30     # cal/mol*K
T = 353.15   # K
KEQ_80 = e**( -(dH/R)*(1/T)+(dS/R) )  # ln(KEQ)=(-dH/R)*(1/T)+dS/R

model.add_ode(
    "s1",
    "-((( k_cyc * KEQ * s3 * s4 * s1 ) / "
    "( 1 + KEQ * s4 )) + k_bkgrnd * s1 )".replace("KEQ", str(KEQ_80)),
)

model.parameters["k_cyc"].prior = mcmc.priors.Uniform(low=1, high=5e3)
model.parameters["k_bkgrnd"].prior = mcmc.priors.Uniform(low=1e-2, high=0.5)

#### HMC run

Finally, the HMC is run to estimate the parameters of the model.

In [ ]:
hmc = mcmc.HMC(
    num_warmup=10000,
    num_samples=10000,
    dt0=0.1,
    max_steps=64**4,
    verbose=1,
    num_chains=8,
    chain_method="parallel",
)

results = hmc.run(model=model, dataset=dataset, yerrs=2.0)

# Apply the median posterior to the model
fitted_model = results.get_fitted_model()

# Print the summary
results.summary()

---

### Results

Catalax provides various plotting functions to visualize the results of the HMC run. Here, the ESS, the corner plot, the trace plot and the fitted model are plotted.

In [ ]:
o1 = "------------------------------------------"
o2 = "Printing the mean of the posterior samples"
o3 = f"k_cyc:    {jnp.median(results.get_samples()['k_cyc'])}"
o4 = f"k_bkgrnd: {jnp.median(results.get_samples()['k_bkgrnd'])}"
o5 = "\n"
print(o1,o5,o2,o5,o5,o3,o5,o4,o5,o1,o5)

In [ ]:
results.plot_ess(show=True)

In [ ]:
results.plot_corner(show=True)

In [ ]:
results.plot_trace(show=True)

In [ ]:
dataset.plot(
    predictor=fitted_model,
    measurement_ids=[m.id for m in dataset.measurements[:3]],
    show=True,
)

#### Save fitted model

The model can be saved as its own data model in JSON format.

In [ ]:
fitted_model.save("./data/output/", "fitted_model")

#### Update EnzymeML document

The EnzymeML document can be updated with the fitted model and the estimated parameters. This allows to keep all information about the experiment in one place and to easily share it with others.

In [ ]:
updated_doc = fitted_model.to_enzymeml(doc)
pe.summary(updated_doc)

In [ ]:
pe.write_enzymeml(updated_doc, path_to_enzymeml)